In [2]:
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (9,4)

In [32]:
patients = pd.read_csv('../data/raw/patients.csv', parse_dates=['registration_date'])
vitals = pd.read_csv('../data/raw/vital_signs.csv', parse_dates=['timestamp'])
history = pd.read_csv('../data/raw/clinical_history.csv')
labs = pd.read_csv('../data/raw/laboratory_results.csv', parse_dates=['timestamp'])
outcomes = pd.read_csv('../data/raw/sepsis_outcomes.csv', parse_dates=['diagnosis_time'])



tables = {'patients': patients, 'vitals': vitals, 'history': history, 'labs': labs, 'outcomes': outcomes}
for name, df in tables.items():
    print(f"{name:10s} Shape={df.shape}")

patients   Shape=(600, 5)
vitals     Shape=(11807, 8)
history    Shape=(1449, 6)
labs       Shape=(2430, 8)
outcomes   Shape=(600, 6)


In [33]:
patients.head()


,patient_id,age,gender,medical_conditions,registration_date
0,1,66,Male,"Diabetes, Cancer (active)",2024-10-19
1,2,42,Female,"Diabetes, Chronic Kidney Disease, Immunosuppre...",2025-06-24
2,3,74,Male,Cancer (active),2024-02-28
3,4,77,Female,"Diabetes, Hypertension",2024-07-31
4,5,25,Female,NaN,2024-07-09


In [9]:
vitals.describe().T

,count,mean,min,25%,50%,75%,max,std
observation_id,11807.0,5904.0,1.0,2952.5,5904.0,8855.5,11807.0,3408.531649
patient_id,11807.0,297.125858,1.0,149.0,294.0,444.0,600.0,172.006979
timestamp,11807,2025-01-05 03:10:59.588379,2024-01-01 04:37:00,2024-07-09 02:07:30,2025-01-15 01:46:00,2025-07-11 20:25:30,2025-12-29 07:13:00,NaN
heart_rate,11490.0,79.892097,43.4,73.2,79.2,85.3,137.4,10.838351
temperature,11514.0,36.84276,34.52,36.55,36.82,37.09,40.25,0.51965
oxygen_saturation,11520.0,97.271389,86.4,96.6,97.4,98.2,100.0,1.555355
respiratory_rate,11519.0,16.543233,8.0,14.6,16.3,18.0,34.6,2.990542
blood_pressure,11545.0,121.214517,55.0,113.8,121.5,129.0,163.9,12.076626


In [10]:
labs.describe().T

,count,mean,min,25%,50%,75%,max,std
lab_id,2430.0,1215.5,1.0,608.25,1215.5,1822.75,2430.0,701.6249
patient_id,2430.0,298.609877,1.0,148.0,300.0,445.0,600.0,172.363294
timestamp,2430,2025-01-01 02:57:24.197530,2024-01-01 12:52:00,2024-07-01 21:03:00,2025-01-10 09:40:00,2025-07-10 09:41:30,2025-12-28 16:48:00,NaN
white_cell_count,2359.0,7.814782,1.0,6.315,7.59,8.97,22.15,2.433649
crp,2370.0,12.306751,0.5,3.325,6.3,9.5,193.1,26.120474
lactate,2356.0,1.141698,0.3,0.8275,1.03,1.23,5.65,0.66575
creatinine,2347.0,0.942833,0.3,0.76,0.91,1.08,2.38,0.286345
platelet_count,2357.0,254.255834,55.0,224.0,255.0,285.0,388.0,46.140953


In [11]:
print('Sepsis Prevalence:', outcomes.sepsis_event.mean().round(3))
outcomes['sepsis_event'].value_counts()

Sepsis Prevalence: 0.12


sepsis_event
False    528
True      72
Name: count, dtype: int64

## Data Quality Assesment

In [14]:
def missing_report (df, name):
    miss = df.isna().mean().mul(100).round(2)
    miss = miss[miss > 0]
    if len(miss):
        print(f'--- {name} ---')
        print(miss.to_string())

for name, df in tables.items():
    missing_report(df, name)


--- patients ---
medical_conditions    25.83
--- vitals ---
heart_rate           2.68
temperature          2.48
oxygen_saturation    2.43
respiratory_rate     2.44
blood_pressure       2.22
--- history ---
diagnosis_history      2.62
medication_history     9.45
treatment_history     14.56
--- labs ---
white_cell_count    2.92
crp                 2.47
lactate             3.05
creatinine          3.42
platelet_count      3.00
--- outcomes ---
diagnosis_time    88.0


In [15]:
for name, df in tables.items():
    print(name, 'duplicate rows', df.duplicated().sum())

patients duplicate rows 0
vitals duplicate rows 0
history duplicate rows 0
labs duplicate rows 0
outcomes duplicate rows 0


In [19]:
#inferencial integrity

valid_ids = set(patients['patient_id'])
for name, df in [('vitals', vitals), ('history', history), ('labs', labs), ('outcomes', outcomes)]:
    orphans = (~df['patient_id'].isin(valid_ids)).sum()
    print(name, 'orphan patient_id rows', orphans)

vitals orphan patient_id rows 0
history orphan patient_id rows 0
labs orphan patient_id rows 0
outcomes orphan patient_id rows 0


In [21]:
ranges = {
    'heart_rate': (30, 200),
    'temperature':(32, 43),
    'oxygen_saturation': (50, 100),
    'respiratory_rate':(5, 60),
    'blood_pressure': (40, 220)
}

for col, (low, high) in ranges.items():
    bad = ((vitals[col] < low) | (vitals[col] > high)).sum()
    print(f'{col}: {bad} out of range')

heart_rate: 0 out of range
temperature: 0 out of range
oxygen_saturation: 0 out of range
respiratory_rate: 0 out of range
blood_pressure: 0 out of range


In [24]:
vitals.columns

Index(['observation_id', 'patient_id', 'timestamp', 'heart_rate',
       'temperature', 'oxygen_saturation', 'respiratory_rate',
       'blood_pressure'],
      dtype='str')

In [25]:
labs.columns

Index(['lab_id', 'patient_id', 'timestamp', 'white_cell_count', 'crp',
       'lactate', 'creatinine', 'platelet_count'],
      dtype='str')

In [35]:
vitals_col =  ['heart_rate',
       'temperature', 'oxygen_saturation', 'respiratory_rate',
       'blood_pressure']

labs_col = ['white_cell_count', 'crp',
       'lactate', 'creatinine', 'platelet_count']

vitals[vitals_col] = vitals.groupby('patient_id')[vitals_col].transform(lambda s: s.ffill())
vitals[vitals_col] = (
    vitals[vitals_col]
    .fillna(
        vitals.groupby('patient_id')[vitals_col]
        .transform('median')
    )
)

labs[labs_col] = labs.groupby('patient_id')[labs_col].transform(lambda s: s.ffill())
labs[labs_col] = (
    labs[labs_col]
    .fillna(
        labs.groupby('patient_id')[labs_col]
        .transform('median')
    )
)

print('Remaining missing values in vitals -> vitals:', vitals[vitals_col].isna().sum().sum(), 
        '& labs', labs[labs_col].isna().sum().sum())


Remaining missing values in vitals -> vitals: 0 & labs 0


In [36]:
patients.to_csv('../data/processed/patients_clean.csv', index=False)
vitals.to_csv('../data/processed/vital_signs_clean.csv', index=False)
history.to_csv('../data/processed/clinical_history_clean.csv', index=False)
labs.to_csv('../data/processed/laboratory_results_clean.csv', index=False)
outcomes.to_csv('../data/processed/sepsis_outcomes_clean.csv', index=False)

## EDA